# ENVIRONMENT SETUP
## Detekcja Środowiska Wykonawczego

In [ ]:
def is_execution_inside_colab():
    try:
        import google.colab
        ! pip install tonic snntorch
        return True
    except:
        return False

## Konfiguracja Specyficzna dla środowiska

In [ ]:
if is_execution_inside_colab():
    datasets_location = "./datasets"
else:
    datasets_location = "../../../datasets"

## Ogólny Setup Środowiska
Konfiguracja odzyskiwania pamięci.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import torch
import tonic
import snntorch
import snntorch.utils as snntorch_utils
import snntorch.functional as snntorch_functional
import matplotlib.pyplot as plt
from torch.utils.checkpoint import checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


device: cuda


# PARAMETER CONFIG
Rozbicie batch_size na batch_size + accum_steps, przez co w tym samym czasie pamięć jest obciążona batch_size / accum_steps zamiast całe batch_size.

n_time_bins do colapsowania zdarzeń pikselowych w pojedyńcze kratki

In [ ]:
CONFIG = {
    "data_root": datasets_location,
    "image_size": 64,
    "batch_size": 8,
    "accum_steps": 2,
    "num_workers": 2,
    "n_time_bins": 10,            # pamiec aktywacji rosnie liniowo z T
    "num_classes": 100,           # usunę BACKGROUND_Google
    "excluded_classes": ["BACKGROUND_Google"], # o tutaj
    "ignore_index": 255,
    "lr": 2e-3,
    "epochs": 5,
    "max_steps_per_epoch": None,
    "validation_split_size": 0.2,
    "use_checkpoint": False,      # mocno zmniejsza potrzeby pamięci
    "use_amp": True,              # save 50% pamięci na fp16 zamiast fp32
}


# Data Augmentation

In [ ]:
import shutil

CACHE_PATH = './cache/ncaltech101'
REBUILD_CACHE = True
if REBUILD_CACHE:
    shutil.rmtree(CACHE_PATH, ignore_errors=True)
os.makedirs(CACHE_PATH, exist_ok=True)

sensor_size = (240, 180, 2)
frame_transform = tonic.transforms.Compose([
    tonic.transforms.Downsample(sensor_size=sensor_size, target_size=CONFIG["image_size"]),
    tonic.transforms.ToFrame(
        sensor_size=(CONFIG["image_size"], CONFIG["image_size"], 2),
        n_time_bins=CONFIG["n_time_bins"],
    ),
    lambda x: torch.from_numpy(x).float()
])

dataset = tonic.datasets.NCALTECH101(save_to='./data', transform=frame_transform)

class FilteredEventDataset(torch.utils.data.Dataset):
    """N-Caltech101 bez podanych klas (np. tla). Zachowuje oryginalne etykiety
    tekstowe - remapping na indeksy dzieje sie pozniej, w decode_targets."""

    def __init__(self, base, excluded_classes):
        self.base = base
        excluded = set(excluded_classes)
        self.keep_indices = [i for i, t in enumerate(base.targets) if t not in excluded]

    def __len__(self):
        return len(self.keep_indices)

    def __getitem__(self, idx):
        return self.base[self.keep_indices[idx]]


print("probek przed filtrowaniem:", len(dataset))
dataset = FilteredEventDataset(dataset, CONFIG["excluded_classes"])
print("probek po usunieciu", CONFIG["excluded_classes"], ":", len(dataset))

cached_dataset = tonic.DiskCachedDataset(dataset, cache_path=CACHE_PATH)

train_size = int((1 - CONFIG["validation_split_size"]) * len(cached_dataset))
test_size = len(cached_dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(cached_dataset, [train_size, test_size])

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=CONFIG["num_workers"] > 0,
)

train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True, **loader_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("train:", len(train_dataset), "test:", len(test_dataset))


  0%|          | 0/3989375649 [00:00<?, ?it/s]

Extracting ./data/NCALTECH101/N-Caltech101-archive.zip to ./data/NCALTECH101
probek przed filtrowaniem: 8709
probek po usunieciu ['BACKGROUND_Google'] : 8242
train: 6593 test: 1649


In [ ]:
from tqdm.auto import tqdm

warmup_loader = torch.utils.data.DataLoader(
    cached_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0
)

for _ in tqdm(warmup_loader, desc="budowanie cache"):
    pass

del warmup_loader
print("cache gotowy:", len(os.listdir(CACHE_PATH)), "plikow")


budowanie cache:   0%|          | 0/1031 [00:00<?, ?it/s]

cache gotowy: 8242 plikow


# Basic Block

In [ ]:
# Surrogate gradient dla propagacji przez nierozniczkowalne spike'i
spike_grad = snntorch.surrogate.atan()


class StatelessLIF(torch.nn.Module):
    """
    Cienka nakladka na snntorch.Leaky z init_hidden=False.
    Żeby checkpointy działały
    """
    def __init__(self, beta, spike_grad):
        super().__init__()
        self.lif = snntorch.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)

    def forward(self, x, mem):
        if mem is None:
            mem = torch.zeros_like(x)
        return self.lif(x, mem)


class SNNBasicBlock(torch.nn.Module):
    """
    Spiking Residual Block - jeden krok czasowy, stan przekazywany jawnie.
    """
    def __init__(self, in_channels, out_channels, downsampling_layer=None, stride=1, beta=0.9):
        super().__init__()

        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(out_channels)
        self.lif1 = StatelessLIF(beta=beta, spike_grad=spike_grad)

        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, stride=1, bias=False)
        self.bn2 = torch.nn.BatchNorm2d(out_channels)
        self.lif2 = StatelessLIF(beta=beta, spike_grad=spike_grad)

        self.downsampling_layer = downsampling_layer

    def forward(self, x, mem1, mem2):
        """
        x: (B, C, H, W) - pojedynczy krok czasowy
        zwraca: (spike'i, nowy mem1, nowy mem2)
        """
        res = x

        out = self.conv1(x)
        out = self.bn1(out)
        spk1, mem1 = self.lif1(out, mem1)

        out = self.conv2(spk1)
        out = self.bn2(out)

        if self.downsampling_layer is not None:
            res = self.downsampling_layer(res)

        out = out + res
        spk2, mem2 = self.lif2(out, mem2)

        return spk2, mem1, mem2


# Konstrukcja Sieci
poprawione padding i stride według oficjalnego [źródła](https://medium.com/@YasinShafiei/residual-networks-resnets-with-implementation-from-scratch-713b7c11f612)

In [ ]:
class SmallSNNResNet(torch.nn.Module):
    """
    Spiking ResNet. Stan neuronow (potencjaly blonowe) jest przekazywany jawnie
    jako plaska lista tensorow, co pozwala bezpiecznie checkpointowac kazdy krok czasowy.
    """
    def __init__(self, num_layers, img_channels=2, num_classes=101, beta=0.9,
                 use_final=True, use_checkpoint=True):
        super().__init__()
        self.expansion = 1
        self.use_final = use_final
        self.use_checkpoint = use_checkpoint
        self.in_channels = 64

        self.conv1 = torch.nn.Conv2d(img_channels, 64, kernel_size=7, stride=2, padding=3, bias=False) # tutaj też poprawnione
        self.bn1 = torch.nn.BatchNorm2d(64)
        self.lif1 = StatelessLIF(beta=beta, spike_grad=spike_grad)
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(SNNBasicBlock, num_layers[0], stride=1, out_channels=64, beta=beta)
        self.layer2 = self._make_layer(SNNBasicBlock, num_layers[1], stride=2, out_channels=128, beta=beta)
        self.layer3 = self._make_layer(SNNBasicBlock, num_layers[2], stride=2, out_channels=256, beta=beta)
        self.layer4 = self._make_layer(SNNBasicBlock, num_layers[3], stride=2, out_channels=512, beta=beta)

        self.avg_pool = torch.nn.AdaptiveAvgPool2d((1, 1))

        if use_final:
            self.fc = torch.nn.Linear(512 * self.expansion, num_classes)
            self.lif_out = StatelessLIF(beta=beta, spike_grad=spike_grad)

        # 1 (stem) + 2 na kazdy blok rezydualny + 1 (wyjscie)
        self.n_states = 1 + 2 * sum(num_layers) + (1 if use_final else 0)

    def _make_layer(self, block, num_layers, stride, out_channels, beta):
        layers = []
        downsampling_layer = None
        if stride != 1 or self.in_channels != out_channels * self.expansion:
            downsampling_layer = torch.nn.Sequential(
                torch.nn.Conv2d(self.in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                torch.nn.BatchNorm2d(out_channels * self.expansion)
            )

        layers.append(block(self.in_channels, out_channels, downsampling_layer, stride, beta=beta))
        self.in_channels = out_channels * self.expansion

        for _ in range(num_layers - 1):
            layers.append(block(self.in_channels, out_channels, beta=beta))

        # ModuleList, nie Sequential - bloki przyjmuja teraz wiecej niz jeden argument
        return torch.nn.ModuleList(layers)

    def forward_single_step(self, x, *mems):
        """
        Jeden krok czasowy. Też coś do checkpointów.
        """
        mems = list(mems)
        i = 0

        out = self.conv1(x)
        out = self.bn1(out)
        spk, mems[i] = self.lif1(out, mems[i]); i += 1
        out = self.maxpool(spk)

        for layer in (self.layer1, self.layer2, self.layer3, self.layer4):
            for block in layer:
                out, mems[i], mems[i + 1] = block(out, mems[i], mems[i + 1])
                i += 2

        out = self.avg_pool(out)
        out = out.flatten(1)

        if self.use_final:
            out = self.fc(out)
            spk_out, mems[i] = self.lif_out(out, mems[i]); i += 1
            return (spk_out, *mems)

        return (out, *mems)

    def forward(self, x_seq):
        """
        x_seq: (T, B, C, H, W) lub (B, T, C, H, W)
        """
        if x_seq.dim() == 5 and x_seq.shape[0] != x_seq.shape[1]:
            # zakladamy (B, T, C, H, W) -> (T, B, C, H, W)
            x_seq = x_seq.transpose(0, 1)

        mems = [None] * self.n_states
        spk_rec = []
        mem_rec = []

        for t in range(x_seq.shape[0]):
            if self.use_checkpoint and self.training and torch.is_grad_enabled():
                outs = checkpoint(self.forward_single_step, x_seq[t], *mems, use_reentrant=False)
            else:
                outs = self.forward_single_step(x_seq[t], *mems)

            spk_out = outs[0]
            mems = list(outs[1:])

            spk_rec.append(spk_out)
            mem_rec.append(mems[-1])   # potencjal blonowy warstwy wyjsciowej

        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)


def SNNResNet18(img_channels=2, num_classes=101, device='cuda', beta=0.9, use_checkpoint=True):
    model = SmallSNNResNet(
        num_layers=[2, 2, 2, 2],
        img_channels=img_channels,
        num_classes=num_classes,
        beta=beta,
        use_checkpoint=use_checkpoint,
    ).to(device=device)
    return model


model = SNNResNet18(device=device, use_checkpoint=CONFIG["use_checkpoint"])

n_params = sum(p.numel() for p in model.parameters())
print(f"parametry: {n_params/1e6:.2f} M")


parametry: 11.23 M


In [ ]:
dataset_root_path = os.path.join('./data', 'NCALTECH101', 'Caltech101')

print(os.listdir(dataset_root_path) )
EXCLUDED_CLASSES = {"BACKGROUND_Google"}

class_names = sorted([
    d for d in os.listdir(dataset_root_path)
    if os.path.isdir(os.path.join(dataset_root_path, d)) and d not in EXCLUDED_CLASSES
])
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

print("liczba klas N-Caltech101 (bez tla):", len(class_names))
assert len(class_names) == CONFIG["num_classes"], (
    "CONFIG['num_classes'] nie jest zbieżna z datasetem, sprawdź czy istnieją `BACKGROUND_Google` and `Faces` "
)


def decode_targets(raw_targets):
    idx = [class_to_idx[t.decode('utf-8') if isinstance(t, bytes) else t] for t in raw_targets]
    return torch.tensor(idx, dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    data, raw_targets = next(iter(train_loader))
    data = data.to(device, non_blocking=True)
    targets = decode_targets(raw_targets) # bye bye dla BACKGROUND_Google
    output_spikes, output_mem = model(data)
    print("spk_rec:", tuple(output_spikes.shape), "| targets:", tuple(targets.shape))

del data, targets, output_spikes, output_mem, raw_targets
torch.cuda.empty_cache()


['strawberry', 'anchor', 'ketch', 'accordion', 'ewer', 'butterfly', 'car_side', 'ant', 'llama', 'schooner', 'Leopards', 'okapi', 'saxophone', 'wrench', 'panda', 'cellphone', 'camera', 'tick', 'windsor_chair', 'trilobite', 'crab', 'helicopter', 'sea_horse', 'beaver', 'wild_cat', 'starfish', 'laptop', 'stegosaurus', 'cougar_face', 'gerenuk', 'emu', 'lobster', 'crocodile_head', 'flamingo', 'metronome', 'rooster', 'BACKGROUND_Google', 'yin_yang', 'scorpion', 'nautilus', 'platypus', 'scissors', 'umbrella', 'ceiling_fan', 'ibis', 'dollar_bill', 'lamp', 'rhino', 'headphone', 'cannon', 'pagoda', 'octopus', 'binocular', 'stapler', 'bonsai', 'buddha', 'airplanes', 'flamingo_head', 'hedgehog', 'joshua_tree', 'chandelier', 'bass', 'inline_skate', 'hawksbill', 'garfield', 'watch', 'Motorbikes', 'water_lilly', 'mandolin', 'dolphin', 'gramophone', 'dragonfly', 'brain', 'sunflower', 'elephant', 'crocodile', 'mayfly', 'pyramid', 'soccer_ball', 'cup', 'minaret', 'revolver', 'ferry', 'lotus', 'stop_sign'

# Petla Treningowa

In [ ]:
import time

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], betas=(0.9, 0.999))
loss_fn = snntorch_functional.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)

scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["use_amp"])
accum_steps = CONFIG["accum_steps"]

loss_hist = []
train_loss_hist = []
test_loss_hist = []
test_acc_hist = []

torch.cuda.reset_peak_memory_stats()
t_start = time.time()

for epoch in range(CONFIG["epochs"]):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    n_batches = 0

    for i, (data, raw_targets) in enumerate(train_loader):
        if CONFIG["max_steps_per_epoch"] is not None and i >= CONFIG["max_steps_per_epoch"]:
            break

        data = data.to(device, non_blocking=True)
        targets = decode_targets(raw_targets)

        with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"]):
            spk_rec, mem_rec = model(data)
            loss_val = loss_fn(spk_rec, targets)

        scaler.scale(loss_val / accum_steps).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss_val.item()
        loss_hist.append(loss_val.item())
        n_batches += 1

        # jawne zwalnianie duzych tensorow przed nastepna iteracja by nie zabierały pamieci
        del data, targets, spk_rec, mem_rec, loss_val

        if i % 25 == 0:
            print(f"Epoch {epoch}, Iteration {i} | Train Loss: {loss_hist[-1]:.3f} "
                  f"| peak GPU: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

    # domkniecie ostatniej, niepelnej akumulacji
    if n_batches % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    train_loss_hist.append(running_loss / max(n_batches, 1))

    model.eval()
    test_loss_total = 0.0
    test_acc_total = 0.0

    with torch.no_grad():
        for data, raw_targets in test_loader:
            data = data.to(device, non_blocking=True)
            targets = decode_targets(raw_targets)

            with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"]):
                spk_rec, mem_rec = model(data)
                loss_val = loss_fn(spk_rec, targets)

            test_loss_total += loss_val.item()
            test_acc_total += snntorch_functional.accuracy_rate(spk_rec, targets)

            del data, targets, spk_rec, mem_rec, loss_val

    avg_test_loss = test_loss_total / len(test_loader)
    avg_test_acc = test_acc_total / len(test_loader)

    test_loss_hist.append(avg_test_loss)
    test_acc_hist.append(avg_test_acc)

    print(f"--- Epoch {epoch} | Train Loss: {train_loss_hist[-1]:.3f} "
          f"| Test Loss: {avg_test_loss:.3f} | Test Acc: {avg_test_acc:.3f}")

elapsed = time.time() - t_start
print(f"\nCzas treningu: {elapsed/60:.1f} min")
print(f"Szczytowe zuzycie pamieci GPU: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")


Epoch 0, Iteration 0 | Train Loss: 0.401 | peak GPU: 0.80 GiB
Epoch 0, Iteration 25 | Train Loss: 0.097 | peak GPU: 0.80 GiB
Epoch 0, Iteration 50 | Train Loss: 0.084 | peak GPU: 0.80 GiB
Epoch 0, Iteration 75 | Train Loss: 0.065 | peak GPU: 0.80 GiB
Epoch 0, Iteration 100 | Train Loss: 0.056 | peak GPU: 0.80 GiB
Epoch 0, Iteration 125 | Train Loss: 0.059 | peak GPU: 0.80 GiB
Epoch 0, Iteration 150 | Train Loss: 0.055 | peak GPU: 0.80 GiB
Epoch 0, Iteration 175 | Train Loss: 0.055 | peak GPU: 0.80 GiB
Epoch 0, Iteration 200 | Train Loss: 0.056 | peak GPU: 0.80 GiB
Epoch 0, Iteration 225 | Train Loss: 0.053 | peak GPU: 0.80 GiB
Epoch 0, Iteration 250 | Train Loss: 0.051 | peak GPU: 0.80 GiB
Epoch 0, Iteration 275 | Train Loss: 0.051 | peak GPU: 0.80 GiB
Epoch 0, Iteration 300 | Train Loss: 0.061 | peak GPU: 0.80 GiB
Epoch 0, Iteration 325 | Train Loss: 0.056 | peak GPU: 0.80 GiB
Epoch 0, Iteration 350 | Train Loss: 0.047 | peak GPU: 0.80 GiB
Epoch 0, Iteration 375 | Train Loss: 0.048 | 

In [ ]:
probe_model = SNNResNet18(device=device, use_checkpoint=True)
probe_optimizer = torch.optim.Adam(probe_model.parameters(), lr=CONFIG["lr"], betas=(0.9, 0.999))
probe_scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["use_amp"])


def snn_step(batch_size):
    x = torch.randn(batch_size, CONFIG["n_time_bins"], 2,
                     CONFIG["image_size"], CONFIG["image_size"], device=device)
    targets = torch.randint(0, CONFIG["num_classes"], (batch_size,), device=device)

    probe_optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"]):
        spk_rec, _ = probe_model(x)
        loss = loss_fn(spk_rec, targets)

    probe_scaler.scale(loss).backward()
    probe_scaler.step(probe_optimizer)
    probe_scaler.update()


for use_ckpt in (True, False):
    probe_model.use_checkpoint = use_ckpt
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for _ in range(20):
        snn_step(CONFIG["batch_size"])

    torch.cuda.synchronize()
    elapsed = time.time() - t0
    print(f"checkpoint={use_ckpt}: {elapsed:.1f}s / 20 krokow, "
          f"peak {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

checkpoint=True: 9.4s / 20 krokow, peak 0.43 GiB
checkpoint=False: 4.4s / 20 krokow, peak 0.47 GiB
